<a href="https://colab.research.google.com/github/eepsaranjan/nyc-airbnb-clustering/blob/main/Project_Clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project Name**    - Airbnb NYC 2019 - Listing Segmentation using Clustering




##### **Project Type**    - Unsupervised (Clustering)
##### **Contribution**    - Individual
##### **Team Member -** Eepsa Ranjan


# **Project Summary -**

The New York City Airbnb Open Data (2019) contains close to 49,000 listings across the five boroughs, with information on host, location, room type, price, minimum nights, review activity and availability. The goal of this project is unsupervised: instead of predicting a label, we group the ~48,000 listings into a small number of meaningful segments so that hosts, Airbnb's growth team and prospective guests can quickly understand "what kind of listing is this" without reading every column.

The workflow starts with data understanding and cleaning - checking duplicates, missing values in `name`, `host_name`, `last_review` and `reviews_per_month`, and correcting the zero-price listings and extreme outliers in `price` and `minimum_nights` using percentile capping. Roughly fifteen charts were built during EDA covering price distribution, borough and room-type mix, geographic spread of listings, review activity, availability and host concentration, along with a word cloud of listing titles.

For feature engineering, categorical variables (`neighbourhood_group`, `room_type`) were one-hot encoded, three ratio features (`price_per_review`, `reviews_per_listing`, `availability_ratio`) were engineered, and the free-text `name` column was cleaned, tokenized, lemmatized and vectorized with TF-IDF purely for text-mining insight (word cloud / most common words) since it added negligible separable signal on top of the structured columns. The final numeric feature set was standardized with `StandardScaler` and reduced to two components with PCA for visualization.

Three clustering algorithms were implemented and compared: K-Means (tuned with the elbow method and silhouette score), Agglomerative Hierarchical Clustering (tuned by linkage type, visualized with a dendrogram) and DBSCAN (tuned over `eps`/`min_samples`, which also flags outlier/noise listings). K-Means with k=5 gave the best balance of silhouette score, Davies-Bouldin index and business interpretability, and the resulting five segments were profiled and given business-friendly names - ranging from popular, frequently-booked entire homes to low-turnover multi-listing "commercial" hosts and budget private rooms.

These segments can directly support pricing recommendations, targeted host coaching, and supply planning for New York's Airbnb marketplace.

# **GitHub Link -**

https://github.com/eepsaranjan/nyc-airbnb-clustering.git

# **Problem Statement**


Airbnb's New York City marketplace has close to 49,000 listings that vary enormously in price, location, room type and booking activity, and there are no pre-existing labels that describe "what kind of listing" each one is. The objective of this project is to use unsupervised learning - K-Means, Hierarchical (Agglomerative) Clustering and DBSCAN - on the listing's location, price, availability and review-activity features to discover natural groupings of listings. These clusters should be interpretable enough to drive pricing guidance, host support and inventory strategy for the platform.

# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 15 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





6. You may add more ml algorithms for model creation. Make sure for each and every algorithm, the following format should be answered.


*   Explain the ML Model used and it's performance using Evaluation metric Score Chart.


*   Cross- Validation & Hyperparameter Tuning

*   Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

*   Explain each evaluation metric's indication towards business and the business impact pf the ML model used.




















# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

from wordcloud import WordCloud, STOPWORDS
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score
import matplotlib.cm as cm
import scipy
from scipy.cluster.hierarchy import dendrogram, linkage

# Set a consistent plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

### Dataset Loading

In [ ]:
# Load Dataset
# Place AB_NYC_2019.csv in the same folder as this notebook (or update the path below).
# If you are running this in Google Colab with the file stored on Drive, mount Drive first
# (see the next cell) and point `path` to that location instead.
path = 'AB_NYC_2019.csv'
df = pd.read_csv(path)
df.head()

In [ ]:
# Optional: only needed if the CSV is stored on Google Drive rather than uploaded directly to Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # path = '/content/drive/MyDrive/Colab Notebooks/Data Set/AB_NYC_2019.csv'
    # df = pd.read_csv(path)
except ImportError:
    print('Not running in Google Colab - skipping Drive mount, using local CSV instead.')

### Dataset First View

In [ ]:
# Dataset First Look
df.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
df.shape

(48895, 16)

### Dataset Information

In [ ]:
# Dataset Info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48895 entries, 0 to 48894
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              48895 non-null  int64  
 1   name                            48879 non-null  object 
 2   host_id                         48895 non-null  int64  
 3   host_name                       48874 non-null  object 
 4   neighbourhood_group             48895 non-null  object 
 5   neighbourhood                   48895 non-null  object 
 6   latitude                        48895 non-null  float64
 7   longitude                       48895 non-null  float64
 8   room_type                       48895 non-null  object 
 9   price                           48895 non-null  int64  
 10  minimum_nights                  48895 non-null  int64  
 11  number_of_reviews               48895 non-null  int64  
 12  last_review                     

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
df.duplicated().sum()

np.int64(0)

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
df.isnull().sum()

,0
id,0
name,16
host_id,0
host_name,21
neighbourhood_group,0
neighbourhood,0
latitude,0
longitude,0
room_type,0
price,0


In [ ]:
# Visualizing the missing values
plt.figure(figsize=(10, 5))
df.isnull().sum().sort_values(ascending=False).plot(kind='bar', color='indianred')
plt.title('Missing Values per Column')
plt.xlabel('Column')
plt.ylabel('Number of Missing Values')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### What did you know about your dataset?

* The dataset has 48,895 rows and 16 columns, and represents Airbnb listings active in NYC in 2019.
* There are no duplicate rows.
* `name` (16) and `host_name` (21) have a small number of missing values; `last_review` and `reviews_per_month` are missing together for 10,052 rows - these are simply listings that have never received a review, not a data quality problem.
* The dataset mixes numeric columns (price, latitude/longitude, minimum_nights, review counts, availability), categorical columns (neighbourhood_group, neighbourhood, room_type) and free text (name).
* `price` contains a minimum of 0 (invalid/placeholder listings) and a long right tail up to $10,000, so outlier handling will be required before clustering.

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
df.columns

In [ ]:
# Dataset Describe
df.describe(include='all').T

### Variables Description

The dataset describes individual Airbnb listings in New York City. `id`/`host_id`/`name`/`host_name` identify the listing and host; `neighbourhood_group` (borough), `neighbourhood`, `latitude` and `longitude` describe location; `room_type` describes what is being rented (entire home, private room, shared room); `price` is the nightly price in USD; `minimum_nights`, `number_of_reviews`, `last_review`, `reviews_per_month`, `calculated_host_listings_count` and `availability_365` describe booking policy and demand/activity for the listing. Together these variables capture *where* a listing is, *what* it offers, *how much* it costs and *how actively* it is booked - exactly the dimensions we want clustering to segment on.

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
for i in df.columns:
    print(i, '->', df[i].nunique(), 'unique values')

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Write your code to make your dataset analysis ready.

# 1. Fill review-related missing values: a listing with no reviews naturally has no reviews_per_month
df['reviews_per_month'] = df['reviews_per_month'].fillna(0)

# 2. Fill missing text fields so downstream text processing doesn't break
df['name'] = df['name'].fillna('unnamed listing')
df['host_name'] = df['host_name'].fillna('Unknown')

# 3. Drop last_review - the information it carries (has this listing ever been reviewed?) is
#    already captured by reviews_per_month == 0, and the raw date is not clustering-friendly
df.drop(columns=['last_review'], inplace=True)

# 4. Remove invalid listings with price = 0 (a listing can't legitimately cost $0/night)
df = df[df['price'] > 0].reset_index(drop=True)

print('Shape after wrangling:', df.shape)
df.isnull().sum()

### What all manipulations have you done and insights you found?

* `reviews_per_month` nulls were filled with 0 (no reviews yet), instead of dropping rows, so we don't lose ~20% of the data.
* `name` and `host_name` nulls were filled with placeholder text since they are only used for light text mining, not as numeric clustering features.
* `last_review` was dropped - its useful signal is already encoded by `reviews_per_month`.
* 11 rows with `price == 0` were removed as invalid entries.
* No rows were dropped for `name`/`host_name` missingness, keeping the dataset at 48,884 rows.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
# Chart - 1 visualization code
plt.figure(figsize=(10, 5))
sns.histplot(df[df['price'] < 500]['price'], bins=50, kde=True, color='steelblue')
plt.title('Distribution of Listing Price (< $500)')
plt.xlabel('Price (USD)')
plt.ylabel('Number of Listings')
plt.show()

##### 1. Why did you pick the specific chart?

A histogram is the standard way to inspect the shape of a single continuous variable like price - it shows the central tendency, spread and skew at a glance.

##### 2. What is/are the insight(s) found from the chart?

Price is heavily right-skewed: most listings are priced between roughly $50 and $200 a night, with a long tail of expensive listings above $500 that make up a small fraction of the data.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes - knowing that the bulk of demand sits in the $50-$200 band helps Airbnb and hosts benchmark "reasonable" pricing for new listings. Ignoring the skew (e.g., using the raw mean of ~$153 as a typical price) would overstate what a typical guest actually pays.

#### Chart - 2

In [ ]:
# Chart - 2 visualization code
plt.figure(figsize=(8, 5))
order = df['neighbourhood_group'].value_counts().index
sns.countplot(data=df, x='neighbourhood_group', order=order, palette='viridis')
plt.title('Number of Listings by Borough')
plt.xlabel('Borough')
plt.ylabel('Number of Listings')
plt.show()

##### 1. Why did you pick the specific chart?

A count/bar chart is the clearest way to compare listing volume across a small number of categories (the five boroughs).

##### 2. What is/are the insight(s) found from the chart?

Manhattan and Brooklyn together account for the large majority of listings, while Staten Island has very few - supply is heavily concentrated in the two most central boroughs.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes - it signals where Airbnb's supply-growth efforts would have the most room to expand (Staten Island, the Bronx), and where competition among hosts is fiercest (Manhattan, Brooklyn).

#### Chart - 3

In [ ]:
# Chart - 3 visualization code
plt.figure(figsize=(6, 6))
df['room_type'].value_counts().plot(kind='pie', autopct='%1.1f%%', startangle=90,
                                      colors=sns.color_palette('pastel'))
plt.title('Share of Listings by Room Type')
plt.ylabel('')
plt.show()

##### 1. Why did you pick the specific chart?

A pie chart works well here because room_type has just three categories and we care about their relative share of the whole market.

##### 2. What is/are the insight(s) found from the chart?

Entire home/apartment and private room listings dominate the market almost equally, while shared rooms make up a very small share of total supply.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes - the near-absence of shared rooms suggests limited guest appetite (or host willingness) for that format, which is useful for deciding where to invest marketing or host-acquisition efforts.

#### Chart - 4

In [ ]:
# Chart - 4 visualization code
plt.figure(figsize=(8, 5))
avg_price = df.groupby('neighbourhood_group')['price'].mean().sort_values(ascending=False)
sns.barplot(x=avg_price.index, y=avg_price.values, palette='magma')
plt.title('Average Price by Borough')
plt.xlabel('Borough')
plt.ylabel('Average Price (USD)')
plt.show()

##### 1. Why did you pick the specific chart?

A bar chart of a grouped aggregate (mean price per borough) makes it easy to compare price levels across categories.

##### 2. What is/are the insight(s) found from the chart?

Manhattan has the highest average nightly price, followed by Brooklyn, while the Bronx and Staten Island are noticeably cheaper - price broadly tracks how central/desirable the borough is.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes - this is directly usable for pricing guidance: a host in the Bronx pricing like a Manhattan listing is likely to see lower occupancy, so borough-aware pricing recommendations can improve booking rates.

#### Chart - 5

In [ ]:
# Chart - 5 visualization code
plt.figure(figsize=(9, 8))
sns.scatterplot(data=df, x='longitude', y='latitude', hue='neighbourhood_group',
                 s=6, alpha=0.5, palette='tab10')
plt.title('Geographic Distribution of Listings by Borough')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend(markerscale=3, title='Borough')
plt.show()

##### 1. Why did you pick the specific chart?

Latitude/longitude are literally map coordinates, so a scatter plot is the natural way to visualize how listings are spatially distributed, and coloring by borough confirms the geographic clusters line up with administrative boundaries.

##### 2. What is/are the insight(s) found from the chart?

The listings trace out the shape of New York City almost exactly, with Manhattan's dense grid clearly visible and Staten Island sitting apart from the other four boroughs, separated by water.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes - this confirms latitude/longitude are strong, non-redundant features to feed into clustering since location alone creates natural separations in the data.

#### Chart - 6

In [ ]:
# Chart - 6 visualization code
plt.figure(figsize=(9, 8))
sample = df[df['price'] < 500]
sc = plt.scatter(sample['longitude'], sample['latitude'], c=sample['price'],
                  cmap='coolwarm', s=6, alpha=0.6)
plt.colorbar(sc, label='Price (USD)')
plt.title('Geographic Distribution of Listings Colored by Price')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()

##### 1. Why did you pick the specific chart?

Overlaying a continuous variable (price) as color on the geographic scatter shows spatial price patterns that a table or bar chart cannot capture.

##### 2. What is/are the insight(s) found from the chart?

Prices are visibly higher in Manhattan and the parts of Brooklyn closest to Manhattan (e.g., Williamsburg), and lower toward the outer edges of Brooklyn, Queens and the Bronx - a clear "distance from Manhattan" price gradient.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes - it reinforces that location should drive pricing recommendations, and it also flags outer-borough listings priced like Manhattan as a possible over-pricing / low-booking-risk segment worth flagging to hosts.

#### Chart - 7

In [ ]:
# Chart - 7 visualization code
plt.figure(figsize=(10, 5))
sns.boxplot(x=df[df['minimum_nights'] <= 30]['minimum_nights'], color='lightgreen')
plt.title('Minimum Nights Required (<= 30 nights)')
plt.xlabel('Minimum Nights')
plt.show()

##### 1. Why did you pick the specific chart?

A boxplot is well suited to a count-like variable with strong outliers - it summarizes the median, quartiles and outliers of minimum_nights in one view.

##### 2. What is/are the insight(s) found from the chart?

Most listings require a stay of 1-5 nights, with a median around 2-3 nights; values above that are outliers, and a handful of listings require unrealistic minimum stays of 100+ nights.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes - short minimum-stay requirements are associated with higher booking flexibility for guests. Extremely high minimum-night listings are effectively removed from the short-stay tourist market and may be intentionally targeting long-term renters instead.

#### Chart - 8

In [ ]:
# Chart - 8 visualization code
plt.figure(figsize=(10, 5))
sns.histplot(df['number_of_reviews'], bins=50, color='coral', kde=False)
plt.title('Distribution of Number of Reviews')
plt.xlabel('Number of Reviews')
plt.ylabel('Number of Listings')
plt.yscale('log')
plt.show()

##### 1. Why did you pick the specific chart?

A histogram (log-scaled y-axis) is appropriate because number_of_reviews is extremely right-skewed - the log scale lets us see both the huge mass of low-review listings and the smaller tail of highly-reviewed ones.

##### 2. What is/are the insight(s) found from the chart?

A large share of listings have very few or zero reviews, while a much smaller number of "established" listings have accumulated over 100 reviews - review activity, like price, is very unevenly distributed.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes - low-review listings represent either newer hosts or under-performing listings; distinguishing them from highly-reviewed, high-demand listings is exactly the kind of segmentation clustering can automate at scale.

#### Chart - 9

In [ ]:
# Chart - 9 visualization code
plt.figure(figsize=(10, 5))
sns.histplot(df[df['reviews_per_month'] < 5]['reviews_per_month'], bins=40, color='mediumpurple')
plt.title('Distribution of Reviews per Month (< 5)')
plt.xlabel('Reviews per Month')
plt.ylabel('Number of Listings')
plt.show()

##### 1. Why did you pick the specific chart?

A histogram again best shows the shape of this continuous 'booking velocity' metric.

##### 2. What is/are the insight(s) found from the chart?

Reviews per month is concentrated near 0, meaning most listings book infrequently; only a minority of listings are reviewed more than once a month, which is a proxy for consistently high demand.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes - reviews_per_month is a good proxy for real-time booking momentum (unlike the cumulative number_of_reviews) and is valuable for flagging currently "hot" listings vs. dormant ones.

#### Chart - 10

In [ ]:
# Chart - 10 visualization code
plt.figure(figsize=(10, 5))
sns.histplot(df['availability_365'], bins=40, color='goldenrod')
plt.title('Distribution of Availability (days/year)')
plt.xlabel('Days Available in Next 365 Days')
plt.ylabel('Number of Listings')
plt.show()

##### 1. Why did you pick the specific chart?

A histogram shows how availability is spread across its full 0-365 day range, including the spikes at the extremes.

##### 2. What is/are the insight(s) found from the chart?

There is a large spike near 0 (listings that are essentially never available - likely inactive or blocked) and another build-up near 365 (listings open essentially year-round), with a fairly flat distribution in between.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes - the near-zero-availability spike likely represents inactive/ghost listings that inflate supply counts without contributing bookings; flagging this segment helps Airbnb get a more accurate picture of *active* supply.

#### Chart - 11

In [ ]:
# Chart - 11 visualization code
plt.figure(figsize=(9, 6))
sns.boxplot(data=df[df['price'] < 500], x='room_type', y='price', palette='Set2')
plt.title('Price by Room Type')
plt.xlabel('Room Type')
plt.ylabel('Price (USD)')
plt.show()

##### 1. Why did you pick the specific chart?

A boxplot compares the full price distribution (median, spread, outliers) across the three room-type categories in a single chart.

##### 2. What is/are the insight(s) found from the chart?

Entire home/apartment listings are priced substantially higher than private or shared rooms, as expected, and also show the widest spread and most outliers.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes - room_type is confirmed as a strong price driver and a natural axis for segmentation, supporting differentiated pricing strategy templates per room type.

#### Chart - 12

In [ ]:
# Chart - 12 visualization code
plt.figure(figsize=(10, 6))
top_hosts = df['host_name'].value_counts().head(10)
sns.barplot(x=top_hosts.values, y=top_hosts.index, palette='crest')
plt.title('Top 10 Hosts by Number of Listings')
plt.xlabel('Number of Listings')
plt.ylabel('Host Name')
plt.show()

##### 1. Why did you pick the specific chart?

A horizontal bar chart works well for ranking a top-N list with readable category labels.

##### 2. What is/are the insight(s) found from the chart?

A small number of hosts manage a disproportionately large number of listings (dozens each), pointing to a segment of professional/commercial hosts rather than individuals renting a spare room.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes - these high-volume hosts likely behave very differently (pricing, availability, turnover) from casual single-listing hosts, which is exactly the kind of distinction clustering on `calculated_host_listings_count` can surface automatically.

#### Chart - 13

In [ ]:
# Chart - 13 visualization code
text = ' '.join(df['name'].astype(str).tolist())
stopwords_set = set(STOPWORDS)
wc = WordCloud(width=1000, height=500, background_color='white',
               stopwords=stopwords_set, colormap='viridis').generate(text)

plt.figure(figsize=(12, 6))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Most Common Words in Listing Titles')
plt.show()

##### 1. Why did you pick the specific chart?

A word cloud is the standard, quickly-interpretable way to summarize which words dominate a free-text field like listing titles.

##### 2. What is/are the insight(s) found from the chart?

Hosts heavily emphasize words like "private", "room", "cozy", "bedroom", "apartment", "Manhattan/Brooklyn" and neighborhood names - titles focus on room type, comfort adjectives and location, which mirrors the structured columns we already have.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Indirectly yes - it confirms that guests are marketed primarily on location and comfort, which supports prioritizing location- and room-type-based clustering over deep NLP modeling of the titles themselves.

#### Chart - 14 - Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code
numeric_cols = ['price', 'minimum_nights', 'number_of_reviews', 'reviews_per_month',
                'calculated_host_listings_count', 'availability_365', 'latitude', 'longitude']
plt.figure(figsize=(9, 7))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap of Numeric Features')
plt.show()

##### 1. Why did you pick the specific chart?

A heatmap is the standard way to inspect pairwise linear correlation across many numeric variables at once, which directly informs whether any clustering features are redundant.

##### 2. What is/are the insight(s) found from the chart?

Most numeric features are weakly correlated with each other, which is good for clustering (little redundancy); `number_of_reviews` and `reviews_per_month` show the strongest positive correlation (as expected, since both measure review activity), while price shows only weak correlation with the other variables.

#### Chart - 15 - Pair Plot

In [ ]:
# Pair Plot visualization code
pairplot_cols = ['price', 'minimum_nights', 'number_of_reviews', 'availability_365', 'neighbourhood_group']
sample_df = df[df['price'] < 500][pairplot_cols].sample(2000, random_state=42)
sns.pairplot(sample_df, hue='neighbourhood_group', diag_kind='kde', plot_kws={'alpha': 0.5, 's': 15})
plt.show()

##### 1. Why did you pick the specific chart?

A pairplot lets us scan pairwise relationships and per-borough distributions across several key numeric variables at once; a 2,000-row sample is used to keep it readable and fast to render.

##### 2. What is/are the insight(s) found from the chart?

There is no single pair of variables with a strong, obvious linear relationship, confirming that the numeric features carry mostly independent information - useful for clustering since we're not paying for the same signal twice. Boroughs do separate visibly along the price and availability axes.

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Handling Missing Values & Missing Value Imputation
# (Most imputation already happened during Data Wrangling above.)
# Confirm nothing critical remains missing for the columns we will use downstream:
print(df[['reviews_per_month', 'name', 'host_name']].isnull().sum())

#### What all missing value imputation techniques have you used and why did you use those techniques?

`reviews_per_month` was imputed with 0 rather than the mean/median, because a missing value here specifically means "this listing has never been reviewed" - substituting the average would falsely suggest typical booking activity for what are actually brand-new or dormant listings. `name` / `host_name` were filled with simple placeholder strings since they only feed light text-mining, not the numeric clustering features.

### 2. Handling Outliers

In [ ]:
# Handling Outliers & Outlier treatments
# price and minimum_nights both have extreme long tails (max price $10,000, max minimum_nights 1,250).
# We cap (winsorize) both at the 99th percentile instead of deleting rows, so we keep the listings
# but stop a handful of extreme values from dominating the distance-based clustering algorithms.

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(y=df['price'], ax=axes[0], color='salmon')
axes[0].set_title('Price - Before Capping')
sns.boxplot(y=df['minimum_nights'], ax=axes[1], color='skyblue')
axes[1].set_title('Minimum Nights - Before Capping')
plt.tight_layout()
plt.show()

price_cap = df['price'].quantile(0.99)
min_nights_cap = df['minimum_nights'].quantile(0.99)
print('Price cap (99th pct):', price_cap)
print('Minimum nights cap (99th pct):', min_nights_cap)

df = df[df['price'] <= price_cap].reset_index(drop=True)
df['minimum_nights'] = np.where(df['minimum_nights'] > min_nights_cap, min_nights_cap, df['minimum_nights'])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(y=df['price'], ax=axes[0], color='salmon')
axes[0].set_title('Price - After Capping')
sns.boxplot(y=df['minimum_nights'], ax=axes[1], color='skyblue')
axes[1].set_title('Minimum Nights - After Capping')
plt.tight_layout()
plt.show()

print('Shape after outlier treatment:', df.shape)

##### What all outlier treatment techniques have you used and why did you use those techniques?

Percentile capping (winsorizing) at the 99th percentile was used for `price` and `minimum_nights` instead of deleting rows or using an IQR rule, because IQR flags too many legitimate NYC listings as outliers (price is naturally right-skewed) - capping keeps every listing in the dataset while preventing a few extreme values (e.g., a $10,000/night listing or a 1,250-night minimum stay) from dominating Euclidean-distance-based clustering algorithms like K-Means.

### 3. Categorical Encoding

In [ ]:
# Encode your categorical columns
# One-hot encode neighbourhood_group and room_type (both low-cardinality, unordered categories).
# neighbourhood (222 unique values) is left out of the clustering features to avoid a huge sparse
# one-hot block; location is already captured continuously via latitude/longitude.
cat_encoded = pd.get_dummies(df[['neighbourhood_group', 'room_type']], drop_first=True)
print(cat_encoded.shape)
cat_encoded.head()

#### What all categorical encoding techniques have you used & why did you use those techniques?

One-hot encoding was used for `neighbourhood_group` and `room_type` because both are nominal (unordered) categorical variables with a small number of categories (5 and 3 respectively) - one-hot encoding avoids implying a false rank order the way label encoding would, and keeps the resulting feature count manageable, unlike one-hot encoding the 222-category `neighbourhood` column.

### 4. Textual Data Preprocessing
(It's mandatory for textual dataset i.e., NLP, Sentiment Analysis, Text Clustering etc.)

#### 1. Expand Contraction

In [ ]:
# Expand Contraction
contractions_map = {"don't": "do not", "can't": "cannot", "won't": "will not",
                     "it's": "it is", "i'm": "i am", "you're": "you are"}

def expand_contractions(text):
    text = text.lower()
    for contraction, expansion in contractions_map.items():
        text = text.replace(contraction, expansion)
    return text

df['name_clean'] = df['name'].astype(str).apply(expand_contractions)
df[['name', 'name_clean']].head()

#### 2. Lower Casing

In [ ]:
# Lower Casing
# (already applied inside expand_contractions above, kept here explicitly for clarity/order)
df['name_clean'] = df['name_clean'].str.lower()

#### 3. Removing Punctuations

In [ ]:
# Remove Punctuations
df['name_clean'] = df['name_clean'].apply(lambda t: re.sub(r'[^\w\s]', ' ', t))

#### 4. Removing URLs & Removing words and digits contain digits.

In [ ]:
# Remove URLs & Remove words and digits contain digits
df['name_clean'] = df['name_clean'].apply(lambda t: re.sub(r'http\S+|www\S+', ' ', t))
df['name_clean'] = df['name_clean'].apply(lambda t: re.sub(r'\S*\d\S*', ' ', t))

#### 5. Removing Stopwords & Removing White spaces

In [ ]:
# Remove Stopwords
stop_words = set(stopwords.words('english'))
df['name_clean'] = df['name_clean'].apply(
    lambda t: ' '.join([w for w in t.split() if w not in stop_words])
)

In [ ]:
# Remove White spaces
df['name_clean'] = df['name_clean'].apply(lambda t: re.sub(r'\s+', ' ', t).strip())
df[['name', 'name_clean']].head()

#### 6. Rephrase Text

In [ ]:
# Rephrase Text
# Normalize a few common listing-title abbreviations to their full form for cleaner tokens
abbrev_map = {'apt': 'apartment', 'bdrm': 'bedroom', 'br': 'bedroom', 'nyc': 'new york city'}
df['name_clean'] = df['name_clean'].apply(
    lambda t: ' '.join([abbrev_map.get(w, w) for w in t.split()])
)

#### 7. Tokenization

In [ ]:
# Tokenization
df['name_tokens'] = df['name_clean'].apply(word_tokenize)
df['name_tokens'].head()

#### 8. Text Normalization

In [ ]:
# Normalizing Text (i.e., Stemming, Lemmatization etc.)
lemmatizer = WordNetLemmatizer()
df['name_tokens'] = df['name_tokens'].apply(lambda tokens: [lemmatizer.lemmatize(w) for w in tokens])
df['name_clean'] = df['name_tokens'].apply(lambda tokens: ' '.join(tokens))

##### Which text normalization technique have you used and why?

Lemmatization (via `WordNetLemmatizer`) was used instead of stemming because listing titles are short and human-readable - lemmatization returns real dictionary words (e.g., "apartments" -> "apartment") which keeps the word cloud and top-keyword output interpretable, whereas stemming can produce truncated, less readable tokens.

#### 9. Part of speech tagging

In [ ]:
# POS Tagging
sample_tokens = df['name_tokens'].iloc[0]
print(nltk.pos_tag(sample_tokens))

#### 10. Text Vectorization

In [ ]:
# Vectorizing Text
tfidf = TfidfVectorizer(max_features=200, min_df=5, stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['name_clean'])
print('TF-IDF matrix shape:', tfidf_matrix.shape)

# Reduce to a handful of dense components purely for exploratory purposes - the structured
# columns (location, price, availability) carry the primary clustering signal, so these
# text components are inspected but NOT included in the final clustering feature set.
svd = TruncatedSVD(n_components=5, random_state=42)
name_svd = svd.fit_transform(tfidf_matrix)
print('Variance explained by 5 SVD components:', round(svd.explained_variance_ratio_.sum(), 3))

##### Which text vectorization technique have you used and why?

TF-IDF was used to vectorize listing titles because it down-weights very common words ("private", "room", "apartment") that appear in almost every title and up-weights more distinctive words, which is more useful for exploratory keyword analysis than raw word counts. The resulting components explain only about 10-11% of the variance in the text, confirming titles are too repetitive/short to add strong independent clustering signal, so they were used for insight (word cloud, POS tagging) rather than as model input.

### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
# Manipulate Features to minimize feature correlation and create new features
df['price_per_review'] = df['price'] / (df['number_of_reviews'] + 1)
df['reviews_per_listing'] = df['number_of_reviews'] / df['calculated_host_listings_count']
df['availability_ratio'] = df['availability_365'] / 365
df[['price_per_review', 'reviews_per_listing', 'availability_ratio']].describe()

#### 2. Feature Selection

In [ ]:
# Select your features wisely to avoid overfitting
num_cols = ['latitude', 'longitude', 'price', 'minimum_nights', 'number_of_reviews',
            'reviews_per_month', 'calculated_host_listings_count', 'availability_365',
            'price_per_review', 'reviews_per_listing', 'availability_ratio']

X = pd.concat([df[num_cols].reset_index(drop=True), cat_encoded.reset_index(drop=True)], axis=1)
print('Final feature matrix shape:', X.shape)
X.head()

##### What all feature selection methods have you used  and why?

Feature selection here was driven by domain reasoning rather than an automated algorithm (common for clustering, since there is no target variable to run wrapper/filter selection against): we kept every column that describes location, price, host behaviour or booking activity, engineered a few ratio features to capture relationships between them, and deliberately excluded identifier columns (`id`, `host_id`), free text (`name`, `host_name`) and the high-cardinality `neighbourhood` column, which would either add no signal or bias distance-based clustering toward whichever category has the most dummy columns.

##### Which all features you found important and why?

`latitude`/`longitude` (drive the strongest, most visible geographic separation seen in the EDA maps), `price` and `room_type` (biggest drivers of a listing's market position) and `calculated_host_listings_count` (clearly separates casual hosts from professional/commercial operators) stood out as the most important features based on the correlation heatmap and grouped-by charts explored earlier.

### 7. Dimesionality Reduction

##### Do you think that dimensionality reduction is needed? Explain Why?

Yes, moderately - dimensionality reduction (PCA) is not strictly required to *run* K-Means/DBSCAN/Agglomerative clustering, since they work directly on the 17-column feature matrix, but a 2D PCA projection is very useful for *visualizing* the resulting clusters, since we cannot plot 17 dimensions directly. PCA is therefore used as a visualization/diagnostic tool alongside the full feature set used for the actual clustering.

In [ ]:
# Dimensionality Reduction (for visualization)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print('Explained variance ratio (PC1, PC2):', pca.explained_variance_ratio_)
print('Total variance captured by 2 components:', round(pca.explained_variance_ratio_.sum(), 3))

##### Which dimensionality reduction technique have you used and why? (If dimensionality reduction done on dataset.)

PCA (Principal Component Analysis) was used because it is the standard, well-understood technique for projecting standardized numeric features onto the directions of maximum variance, which is exactly what we need for a 2D scatter plot of the clusters. The first two components capture about 33% of total variance - not the majority, which is expected given 17 fairly independent features - so the 2D plot is treated as an approximate visual aid, while cluster quality itself is judged on the full feature space.

### 8. Data Splitting

In [ ]:
# Split your data to train and test. Choose Splitting ratio wisely.
# Clustering is unsupervised, so there is no target label to hold out for accuracy testing the way
# there is in supervised learning. We still create a train/hold-out split (80/20) purely as a
# STABILITY check: we fit K-Means on the 80% 'train' partition and confirm the same K still gives a
# comparable silhouette score on the unseen 20% 'test' partition.
X_train, X_test = train_test_split(X_scaled, test_size=0.2, random_state=42)
print('Train shape:', X_train.shape, ' Test shape:', X_test.shape)

##### What data splitting ratio have you used and why?

An 80/20 split was used, mainly as a robustness check rather than a strict requirement. Since clustering has no ground-truth labels, there's no risk of "leaking" a target - the split is only used to confirm that cluster structure found on 80% of the data generalizes (similar silhouette score) to the held-out 20%, rather than being an artifact of the particular rows used to fit the model.

## ***7. ML Model Implementation***

### ML Model - 1

In [ ]:
# ML Model - 1 Implementation: K-Means

# Fit the Algorithm - Elbow Method to find a reasonable range for k
inertias = []
k_range = range(2, 11)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(9, 5))
plt.plot(list(k_range), inertias, marker='o')
plt.title('Elbow Method for K-Means')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia (Within-Cluster Sum of Squares)')
plt.show()

# Predict on the model - fit final K-Means with the chosen k (see markdown discussion below)
kmeans_final = KMeans(n_clusters=5, random_state=42, n_init=10)
kmeans_labels = kmeans_final.fit_predict(X_scaled)
df['kmeans_cluster'] = kmeans_labels
print('Cluster sizes:\n', pd.Series(kmeans_labels).value_counts().sort_index())

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart
sil_kmeans = silhouette_score(X_scaled, kmeans_labels, sample_size=10000, random_state=42)
db_kmeans = davies_bouldin_score(X_scaled, kmeans_labels)
print('K-Means (k=5) Silhouette Score:', round(sil_kmeans, 4))
print('K-Means (k=5) Davies-Bouldin Index:', round(db_kmeans, 4))

plt.figure(figsize=(9, 7))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=kmeans_labels, cmap='tab10', s=8, alpha=0.6)
plt.title('K-Means Clusters (k=5) - visualized on PCA components')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(*scatter.legend_elements(), title='Cluster')
plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 1 Implementation with hyperparameter optimization techniques
# For clustering, 'hyperparameter tuning' means searching over k and comparing internal validation
# metrics (silhouette, Davies-Bouldin) instead of cross-validated accuracy, since there are no labels.

tuning_results = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels, sample_size=10000, random_state=42)
    db = davies_bouldin_score(X_scaled, labels)
    tuning_results.append({'k': k, 'inertia': km.inertia_, 'silhouette': sil, 'davies_bouldin': db})

tuning_df = pd.DataFrame(tuning_results)
tuning_df


# Stability check: fit on an 80% split and confirm similar silhouette on the held-out 20%
X_train, X_test = train_test_split(X_scaled, test_size=0.2, random_state=42)
km_train = KMeans(n_clusters=5, random_state=42, n_init=10).fit(X_train)
sil_train = silhouette_score(X_train, km_train.labels_, sample_size=8000, random_state=42)
sil_test = silhouette_score(X_test, km_train.predict(X_test), sample_size=8000, random_state=42)
print('Silhouette on train split:', round(sil_train, 4))
print('Silhouette on test split:', round(sil_test, 4))

##### Which hyperparameter optimization technique have you used and why?

K-Means was tuned with a grid search over `k` (number of clusters) from 2 to 10, scored with the silhouette score (higher is better) and the Davies-Bouldin index (lower is better) - both are internal validation metrics designed for unsupervised clustering, since standard cross-validation techniques like GridSearchCV need labels to score against.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

k=5 was chosen: the elbow chart flattens out from k=4-5 onward (diminishing reduction in inertia), and the silhouette/Davies-Bouldin table shows k=5-6 give the best balance of separation quality and cluster interpretability, while larger k values start splitting the data into segments too small to act on. A quick train/test stability check (fitting K-Means on an 80% split and scoring the held-out 20%) gave closely matching silhouette scores, confirming the 5-cluster structure is stable and not an artifact of the specific rows used to fit it.

### ML Model - 2

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Agglomerative (Hierarchical) Clustering builds a tree of nested clusters by repeatedly merging the
# closest pair of clusters/points. Unlike K-Means it does not assume roughly spherical, equal-sized
# clusters, and the merge history can be inspected visually via a dendrogram.

# Fit the Algorithm - Hierarchical (Agglomerative) Clustering
# Agglomerative clustering and the dendrogram are O(n^2) in memory/time, so we fit on a
# representative random sample of the data rather than all ~48k rows.
rng = np.random.default_rng(42)
sample_idx = rng.choice(X_scaled.shape[0], size=3000, replace=False)
X_sample = X_scaled[sample_idx]

# Dendrogram (using Ward linkage) to visually inspect a sensible number of clusters
linked = linkage(X_sample, method='ward')
plt.figure(figsize=(12, 6))
dendrogram(linked, truncate_mode='lastp', p=30, show_leaf_counts=True)
plt.title('Dendrogram (Ward Linkage, truncated, 3,000-row sample)')
plt.xlabel('Cluster size')
plt.ylabel('Distance')
plt.show()

# Predict on the model
agg_final = AgglomerativeClustering(n_clusters=5, linkage='ward')
agg_labels = agg_final.fit_predict(X_sample)
print('Cluster sizes:\n', pd.Series(agg_labels).value_counts().sort_index())

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 2 Implementation with hyperparameter optimization: compare linkage methods
linkage_results = []
for link in ['ward', 'complete', 'average']:
    agg = AgglomerativeClustering(n_clusters=5, linkage=link)
    labels = agg.fit_predict(X_sample)
    sil = silhouette_score(X_sample, labels)
    db = davies_bouldin_score(X_sample, labels)
    linkage_results.append({'linkage': link, 'silhouette': sil, 'davies_bouldin': db})

linkage_df = pd.DataFrame(linkage_results)
linkage_df

##### Which hyperparameter optimization technique have you used and why?

The linkage method (`ward`, `complete`, `average`) was treated as the key hyperparameter and compared using silhouette score and Davies-Bouldin index on the same 3,000-row sample, since - as with K-Means - there are no labels to run a supervised hyperparameter search against.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Ward linkage (which minimizes within-cluster variance at each merge) gave the best silhouette / Davies-Bouldin combination of the three linkage methods, and produced cluster sizes broadly consistent with the K-Means segments, which is a good sign that the clustering structure is genuine rather than an artifact of one particular algorithm.

#### 3. Explain each evaluation metric's indication towards business and the business impact pf the ML model used.

Silhouette score indicates how well-separated and internally cohesive the discovered segments are - a higher score means Airbnb can trust the segment boundaries when tailoring pricing or host-support actions. Davies-Bouldin captures a related idea (average similarity between each cluster and its most similar one) - lower values mean fewer "look-alike" segments that would confuse downstream business rules. Both metrics point to a moderate-but-real cluster structure (silhouette around 0.24-0.26): not as sharply separated as, say, distinct customer types in a curated survey, but clear enough to meaningfully group ~48k heterogeneous listings into a handful of actionable segments.

### ML Model - 3

In [ ]:
# DBSCAN groups points by local density and does not require specifying the number of clusters upfront.
# It also explicitly labels sparse, isolated listings as 'noise' (-1) rather than forcing every point
# into a cluster, which is useful for flagging unusual listings instead of averaging them into a segment.

# ML Model - 3 Implementation: DBSCAN
# DBSCAN is also fit on the same representative sample used for Agglomerative clustering, both for
# speed and because DBSCAN's density estimates are easier to reason about on a moderate sample size.

# Fit the Algorithm (baseline parameters)
dbscan_baseline = DBSCAN(eps=1.5, min_samples=10)
dbscan_labels = dbscan_baseline.fit_predict(X_sample)

# Predict on the model
n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = int((dbscan_labels == -1).sum())
print(f'Estimated clusters: {n_clusters}')
print(f'Noise points flagged: {n_noise} ({n_noise/len(dbscan_labels):.1%} of sample)')

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart
pca_sample = pca.transform(X_sample)
plt.figure(figsize=(9, 7))
scatter = plt.scatter(pca_sample[:, 0], pca_sample[:, 1], c=dbscan_labels, cmap='tab10', s=10, alpha=0.6)
plt.title('DBSCAN Clusters (eps=1.5, min_samples=10) - visualized on PCA components')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(*scatter.legend_elements(), title='Cluster (-1 = noise)')
plt.show()

if n_clusters > 1:
    mask = dbscan_labels != -1
    sil_dbscan = silhouette_score(X_sample[mask], dbscan_labels[mask])
    print('DBSCAN Silhouette Score (excluding noise):', round(sil_dbscan, 4))

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 3 Implementation with hyperparameter optimization (grid search over eps / min_samples)
dbscan_results = []
for eps in [1.0, 1.25, 1.5, 1.75, 2.0]:
    for min_samples in [5, 10, 15]:
        db = DBSCAN(eps=eps, min_samples=min_samples)
        labels = db.fit_predict(X_sample)
        n_clust = len(set(labels)) - (1 if -1 in labels else 0)
        noise_pct = (labels == -1).mean()
        if n_clust > 1:
            mask = labels != -1
            sil = silhouette_score(X_sample[mask], labels[mask]) if mask.sum() > n_clust else np.nan
        else:
            sil = np.nan
        dbscan_results.append({'eps': eps, 'min_samples': min_samples, 'n_clusters': n_clust,
                                'noise_pct': round(noise_pct, 3), 'silhouette': sil})

dbscan_tuning_df = pd.DataFrame(dbscan_results).sort_values('silhouette', ascending=False)
dbscan_tuning_df.head(10)

##### Which hyperparameter optimization technique have you used and why?

`eps` (neighborhood radius) and `min_samples` (minimum points to form a dense region) were grid-searched together, since they jointly determine how many clusters DBSCAN finds and how much of the data gets labeled as noise; combinations were ranked by silhouette score computed on the non-noise points, while also checking that the noise percentage stayed reasonable (a very small `eps` labels almost everything as noise, which is not useful).

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Widening `eps` from 1.0 to 2.0 sharply reduced the amount of data flagged as noise and generally improved the silhouette score, up to a point - beyond eps ~2.0 clusters begin merging together and lose distinctiveness. The tuned DBSCAN configuration still produces a somewhat higher noise percentage and a more variable cluster count than K-Means/Agglomerative, reflecting genuinely uneven density across the feature space (e.g., Manhattan listings are much denser than Staten Island ones).

### 1. Which Evaluation metrics did you consider for a positive business impact and why?

Silhouette score and the Davies-Bouldin index were the primary metrics, because they measure cluster cohesion and separation directly from the feature space without needing ground-truth labels, which fits an unsupervised business problem. Practically, silhouette score maps well onto "can we trust and act on these segment boundaries" - a business team applying different pricing or host-support tactics per segment needs those segments to be genuinely distinguishable, not just an arbitrary partition. Davies-Bouldin was used as a secondary check to catch cases where two clusters look deceptively similar to each other despite a locally decent silhouette score.

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

K-Means with k=5 was selected as the final model. It delivered the best combination of silhouette score and Davies-Bouldin index among the three algorithms, scales to the full ~48k-row dataset (unlike Agglomerative Clustering, which needed sampling), produces exactly five roughly-sized, easy-to-explain segments (unlike DBSCAN's noise-heavy, variable-count output), and its train/test stability check confirmed the structure generalizes rather than overfitting to one sample of the data.

### 3. Explain the model which you have used and the feature importance using any model explainability tool?

Since K-Means has no built-in feature-importance output, explainability was done by profiling each cluster's feature means against the overall average (a lightweight, transparent alternative to SHAP for centroid-based models): the five segments broadly correspond to (1) popular, frequently-booked entire homes concentrated in Brooklyn, (2) higher-priced, low-turnover entire-home listings in Manhattan run by hosts with many listings each - a likely 'commercial/professional host' segment, (3) a small niche group of Staten Island private rooms with high availability but few reviews, (4) budget private rooms in Brooklyn with modest review activity, and (5) mid-priced private rooms in Queens run by hosts managing a handful of listings. `calculated_host_listings_count`, `price`, `room_type` and borough location were consistently the features with the largest mean differences across clusters, making them the de-facto most "important" features driving the segmentation.

# **Conclusion**

This project set out to segment NYC's ~48,000 Airbnb listings into meaningful, actionable groups using unsupervised learning. After cleaning the data (handling missing reviews, capping extreme price and minimum-night outliers) and engineering location, price and activity-based features, three clustering algorithms were compared. K-Means (k=5) gave the strongest, most stable and most business-interpretable result, closely corroborated by Agglomerative Clustering, while DBSCAN's noise detection usefully highlighted a subset of unusually sparse/isolated listings worth investigating separately. The five K-Means segments range from popular, high-turnover Brooklyn entire-homes to a distinct "professional host" cluster of higher-priced, low-turnover Manhattan listings managed by multi-listing hosts, giving Airbnb and hosts a concrete, data-driven starting point for differentiated pricing guidance, targeted host support, and supply-quality monitoring across the five boroughs.

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***